# 03 — Preprocessing & Feature Selection
## India Tourist Crowd Forecasting System

### What this notebook does
Cleans, encodes and selects the final 59 features from the 450,540-row timeseries dataset.

**Pipeline:**
1. **Step 3 — Encoding:** One-hot + frequency encoding of categorical columns
2. **Step 4A — Feature Exclusions:** Remove leakage, near-constant, duplicate, metadata columns
3. **Step 4B — Importance Pruning:** XGBoost importance → drop low-signal features
4. **Interactions:** Add 9 place-type × season interaction features
5. **Final Output:** 59 features ready for model training

**Input:** `india_crowd_timeseries_v2.csv` (450,540 × 106)
**Output:** `india_tourist_crowd_forecast_dataset.csv` + `india_tourist_crowd_features.json`

In [ ]:
# ── Setup ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, json
import warnings
warnings.filterwarnings('ignore')

# Load timeseries dataset
df_exp = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/india_crowd_timeseries_v2 (1).csv')
print(f'✅ Loaded: {df_exp.shape}')
print(f'place_id present: {"place_id" in df_exp.columns}')
print(f'relative_crowd_label present: {"relative_crowd_label" in df_exp.columns}')
print(f'\nLabel distribution:')
print(df_exp['relative_crowd_label'].value_counts(normalize=True).round(3))

## Step 3 — Encoding

In [ ]:
# ─────────────────────────────────────────────────
# STEP 3: ENCODING
# ─────────────────────────────────────────────────
print(f'BEFORE encoding: {df_exp.shape}')

# Drop non-predictive columns
DROP_COLS = ['place_name', 'last_refreshed_date']
df_model = df_exp.drop(columns=[c for c in DROP_COLS if c in df_exp.columns])
print(f'[Drop] Removed: {DROP_COLS}')

# One-hot encode low-cardinality categoricals
ONEHOT_COLS = ['place_type','season','real_weather_category',
               'zone','weekly_off_real','peak_day','pt_source']
ONEHOT_COLS = [c for c in ONEHOT_COLS if c in df_model.columns]
for c in ONEHOT_COLS: df_model[c] = df_model[c].fillna('None')
df_model = pd.get_dummies(df_model, columns=ONEHOT_COLS, drop_first=False)
print(f'[One-Hot] Encoded: {ONEHOT_COLS}')

# Binary encode source
if 'source' in df_model.columns:
    df_model['source_google_api'] = (df_model['source']=='google_places_api').astype(int)
    df_model = df_model.drop(columns=['source'])

# Frequency encode high-cardinality cols
for col in ['city', 'state']:
    if col in df_model.columns:
        freq = df_model[col].value_counts(normalize=True)
        df_model[f'{col}_freq'] = df_model[col].map(freq)
print(f'[Frequency] city_freq, state_freq added')

# Festival name frequency
if 'major_festival_name' in df_model.columns:
    df_model['major_festival_name'] = df_model['major_festival_name'].fillna('No_Festival')
    freq = df_model['major_festival_name'].value_counts(normalize=True)
    df_model['festival_name_freq'] = df_model['major_festival_name'].map(freq)

# Drop raw text cols
TEXT_DROP = ['city','state','major_festival_name']
df_model = df_model.drop(columns=[c for c in TEXT_DROP if c in df_model.columns])

# Convert bool to int
bool_cols = df_model.select_dtypes(include=['bool']).columns.tolist()
df_model[bool_cols] = df_model[bool_cols].astype(int)

# Memory optimization
mem_before = df_model.memory_usage(deep=True).sum()/1e6
for col in df_model.select_dtypes(include=['float64']).columns:
    df_model[col] = pd.to_numeric(df_model[col], downcast='float')
for col in df_model.select_dtypes(include=['int64']).columns:
    df_model[col] = pd.to_numeric(df_model[col], downcast='integer')
mem_after = df_model.memory_usage(deep=True).sum()/1e6

print(f'\n✅ ENCODING COMPLETE')
print(f'   Shape : {df_model.shape}')
print(f'   Memory: {mem_before:.1f} MB → {mem_after:.1f} MB')

ENCODED_PATH = '/content/drive/MyDrive/Colab Notebooks/india_crowd_encoded_v3.csv'
df_model.to_csv(ENCODED_PATH, index=False)
print(f'💾 Saved → {ENCODED_PATH}')

## Step 4A — Feature Exclusions + Variance Check

In [ ]:
# ─────────────────────────────────────────────────
# STEP 4A: Explicit feature exclusions
# ─────────────────────────────────────────────────
df_model = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/india_crowd_encoded_v3.csv')
print(f'Loaded encoded dataset: {df_model.shape}')

TARGET_COLS    = ['monthly_crowd_label','relative_crowd_label','monthly_crowd_score']
ID_COLS        = ['place_id']
LEAK_COLS      = ['festival_boost_score','calendar_boost','weather_score','is_low_crowd_alt']
NEAR_CONSTANT  = ['night_avg','peak_is_weekend','airport_nearby','climate_data_real',
                  'best_morning','best_afternoon','best_evening']
DUPLICATES     = ['baseline_num_reviews']
METADATA       = ['has_real_hours','trend_data_available','num_reviews_corrected',
                  'age_known','source_google_api']
TIME_OF_DAY    = ['morning_avg','afternoon_avg','evening_avg',
                  'morning_avg_norm','afternoon_avg_norm','evening_avg_norm']
DAY_OF_WEEK    = ['mon_avg','tue_avg','wed_avg','thu_avg','fri_avg','sat_avg','sun_avg']
REDUNDANT_BUSY = ['busyness_max','peak_hour','busyness_avg_norm',
                  'weekday_avg_norm','weekend_avg_norm']

ALL_EXCLUDE = set(TARGET_COLS+ID_COLS+LEAK_COLS+NEAR_CONSTANT+DUPLICATES+
                  METADATA+TIME_OF_DAY+DAY_OF_WEEK+REDUNDANT_BUSY)

feature_cols = [c for c in df_model.columns if c not in ALL_EXCLUDE]
print(f'Features remaining: {len(feature_cols)}')

# Variance threshold (>99% same value)
low_var_drop = []
for c in feature_cols:
    if df_model[c].dtype in [np.float32,np.float64,np.int32,np.int64,'int8','uint8']:
        top_val_pct = df_model[c].value_counts(normalize=True).iloc[0]
        if top_val_pct > 0.99:
            low_var_drop.append(c)

if low_var_drop:
    print(f'Low variance dropped: {low_var_drop}')
    feature_cols = [c for c in feature_cols if c not in low_var_drop]

# Fix: swap norm/log for originals
for col in ['rating_norm','time_needed_hrs_norm','popularity_score_log']:
    if col in feature_cols: feature_cols.remove(col)
for col in ['rating','time_needed_hrs']:
    if col in df_model.columns and col not in feature_cols: feature_cols.append(col)

# Swap place_type_* dummies for is_* flags
place_type_dummies = [c for c in feature_cols if c.startswith('place_type_')]
for c in place_type_dummies:
    flag = f'is_{c.replace("place_type_","").lower().replace(" ","_")}'
    if flag in df_model.columns:
        if c in feature_cols: feature_cols.remove(c)
        if flag not in feature_cols: feature_cols.append(flag)

# Drop year, pt_source_synthetic, lat/lng, weekly_off_real_Unknown
for drop in ['year','pt_source_synthetic','latitude','longitude','weekly_off_real_Unknown']:
    if drop in feature_cols: feature_cols.remove(drop)

print(f'\nFinal feature count after 4A: {len(feature_cols)}')

with open('/content/drive/MyDrive/Colab Notebooks/feature_cols_v1.json','w') as f:
    json.dump(feature_cols, f)
print(f'💾 Saved → feature_cols_v1.json')

## Step 4B — Feature Importance Pruning (XGBoost)

In [ ]:
# ─────────────────────────────────────────────────
# STEP 4B: Feature Importance Pruning
# Train lightweight XGBoost on 30% sample
# Drop features with importance < 0.001
# ─────────────────────────────────────────────────
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

with open('/content/drive/MyDrive/Colab Notebooks/feature_cols_v1.json') as f:
    feature_cols = json.load(f)

TARGET = 'relative_crowd_label'
le = LabelEncoder()
y_all = le.fit_transform(df_model[TARGET])
print(f'Classes: {dict(enumerate(le.classes_))}')

# 30% sample
sample_idx = df_model.sample(frac=0.30, random_state=42).index
feat_in_model = [c for c in feature_cols if c in df_model.columns]
X_sample = df_model.loc[sample_idx, feat_in_model].fillna(0)
y_sample  = y_all[sample_idx]
groups    = df_model.loc[sample_idx, 'place_id'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_sample, y_sample, groups=groups))
X_train = X_sample.iloc[train_idx]; X_test = X_sample.iloc[test_idx]
y_train = y_sample[train_idx];      y_test = y_sample[test_idx]

print(f'Training XGBoost on 30% sample: {X_train.shape}')
model_imp = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, eval_metric='mlogloss',
    random_state=42, n_jobs=-1, verbosity=0)
model_imp.fit(X_train, y_train)

acc = accuracy_score(y_test, model_imp.predict(X_test))
print(f'Quick accuracy (30% sample): {acc*100:.2f}%')

importances = pd.Series(
    model_imp.feature_importances_, index=feat_in_model
).sort_values(ascending=False)

print(f'\nTop 20 features:')
for feat, imp in importances.head(20).items():
    bar = '█' * int(imp * 500)
    print(f'  {feat:<40} {imp:.4f}  {bar}')

# Drop low-importance features
THRESHOLD = 0.001
drop_low = importances[importances < THRESHOLD].index.tolist()
final_features = [c for c in feat_in_model if c not in drop_low]
print(f'\nDropped {len(drop_low)} features (importance < {THRESHOLD})')
print(f'Final features: {len(final_features)}')

# Feature importance plot
fig, ax = plt.subplots(figsize=(10, 8))
importances.head(30).sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(x=THRESHOLD, color='red', linestyle='--', label=f'Threshold ({THRESHOLD})')
ax.set_title('Top 30 Feature Importances', fontweight='bold', fontsize=13)
ax.set_xlabel('XGBoost Importance Score')
ax.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Colab Notebooks/feature_importances.png',
            dpi=150, bbox_inches='tight')
plt.show()

with open('/content/drive/MyDrive/Colab Notebooks/feature_cols_final.json','w') as f:
    json.dump(final_features, f)
print(f'💾 Saved → feature_cols_final.json + feature_importances.png')

## Final Step — Add 9 Interaction Features + Save Final Files

In [ ]:
# ─────────────────────────────────────────────────
# FINAL: Add interaction features + rename + save
# ─────────────────────────────────────────────────
import os
from google.colab import files

df_model = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/india_crowd_encoded_v3.csv')
with open('/content/drive/MyDrive/Colab Notebooks/feature_cols_final.json') as f:
    features = json.load(f)

print(f'Before: {df_model.shape}, {len(features)} features')

# Add 9 interaction features (place type × season)
df_model['beach_x_winter']                = df_model['is_beach'] * df_model['season_Winter']
df_model['beach_x_school_vacation']       = df_model['is_beach'] * df_model['is_school_vacation']
df_model['wildlife_x_post_monsoon']       = df_model.get('is_wildlife', 0) * df_model['season_Post-Monsoon']
df_model['nature_x_post_monsoon']         = df_model.get('is_nature', 0) * df_model['season_Post-Monsoon']
df_model['heritage_x_festival']           = df_model['is_heritage'] * df_model['has_major_festival']
df_model['heritage_x_winter']             = df_model['is_heritage'] * df_model['season_Winter']
df_model['museum_x_monsoon']              = df_model['is_museum'] * df_model['season_Monsoon']
df_model['park_x_school_vacation']        = df_model['is_park'] * df_model['is_school_vacation']
df_model['hillstation_x_school_vacation'] = df_model.get('is_hill_station', 0) * df_model['is_school_vacation']

NEW_FEATURES = [
    'beach_x_winter','beach_x_school_vacation',
    'wildlife_x_post_monsoon','nature_x_post_monsoon',
    'heritage_x_festival','heritage_x_winter',
    'museum_x_monsoon','park_x_school_vacation',
    'hillstation_x_school_vacation'
]
features_final = list(dict.fromkeys(features + NEW_FEATURES))
print(f'After : {df_model.shape}, {len(features_final)} features')

# Save with final names
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/india_tourist_crowd_forecast_dataset.csv'
FEAT_PATH = '/content/drive/MyDrive/Colab Notebooks/india_tourist_crowd_features.json'

df_model.to_csv(DATA_PATH, index=False)
with open(FEAT_PATH, 'w') as f:
    json.dump(features_final, f)

print(f'\n✅ PREPROCESSING COMPLETE')
print(f'   Dataset : {df_model.shape}')
print(f'   Features: {len(features_final)} (50 base + 9 interactions)')
print(f'   Target  : relative_crowd_label (Low/Medium/High)')
print(f'💾 {DATA_PATH}')
print(f'💾 {FEAT_PATH}')

# Download
files.download(DATA_PATH)
files.download(FEAT_PATH)
print('✅ Downloaded!')